[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/labs/L3/lab.ipynb)

# Lab 3 · Árboles, bosques y boosting
**DAMMG · Módulo VI · Sesión 3** · unos 55 min de trabajo · entrega en Moodle: **dom 15 nov, 23:59** (tolerancia hasta el mié 18 nov)

**Al terminar podrás**
- **(O4.1)** calcular la entropía, el Gini y la ganancia de información de un corte, y elegir el mejor corte de una variable;
- **(O4.2)** comparar un árbol, un bosque aleatorio y un boosting en validación, y atribuir las diferencias a sesgo o varianza;
- detectar una variable que "ve el futuro" antes de que infle tus métricas.

**Ruta (PRIMM).** L0 preparar, L1 ejemplo resuelto (¿gato o no gato?), L2 investigar la profundidad con un slider, L3 completar funciones, L4 cazar la fuga, L5 MiBici: árbol vs bosque vs boosting, L6 mini-reto.

> **Carril exprés (L3, L5 y L6).** Si ya dominas los árboles, completa L3 (es lo que se califica) y salta a L5–L6. Ejecuta igual **todas** las celdas en orden aunque no las leas: varias autocomprobaciones usan variables de celdas anteriores.

**Qué se califica.** Las celdas `graded`: `entropia`, `ganancia_info` y `mejor_corte` (L3). Reglas: nombres y argumentos intactos; dentro de las funciones solo `numpy` y `math`, sin variables globales, sin `du.` y sin funciones de otras celdas (el evaluador solo ve las celdas `graded`; la autocomprobación te avisa si algo se escapa). **Cuánto vale:** cada lab es 7.5 % del módulo; tu nota es la fracción de pruebas del evaluador que pasan tus funciones.

**Fuentes.** Andrew Ng, *Machine Learning Specialization*, Curso 2, semana 4 (árboles de decisión, entropía, ganancia de información, variables continuas, bosques y boosting). Stanford CS229 (2018), clase 9. Cornell CS4780, clases de árboles y ensambles.

## L0 · Preparación

> **Antes de escribir nada**
> 1. *Archivo*, opción *Guardar una copia en Drive* y trabaja en esa copia; si no, al cerrar la pestaña pierdes todo.
> 2. Ejecuta las celdas **en orden**. Si reinicias el entorno o algo falla raro: *Entorno de ejecución*, opción *Reiniciar y ejecutar todo*.
> 3. Escribe dentro de las celdas que ya existen (no copies/pegues celdas): así no pierden la etiqueta que usa el evaluador.
>
> *Palabras en inglés que usamos tal cual:* **baseline** (línea base: el modelo tonto que hay que vencer) · **pipeline** (cadena de preprocesamiento + modelo) · **proxy** (variable sustituta de lo que de verdad queremos medir) · **bootstrap** (remuestreo con reemplazo) · **chat waterfall** (todos escriben su respuesta en el chat y la envían al mismo tiempo) · **precision** y **recall** (los nombres de sklearn para precisión y exhaustividad).

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES

du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

> ### ✅ Reacción en Zoom · fin de L0
> Reacciona con ✅ cuando veas `✅ entorno listo`.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L1 · Ejemplo resuelto: ¿gato o no gato?
Un refugio de animales registró 10 ingresos con tres rasgos fáciles de observar y el peso. Queremos un árbol que diga si un animal es gato ($y=1$). Los datos son inventados para este lab.

La entropía de un nodo mide su "revoltura" en bits: $H = -\sum_k p_k\log_2 p_k$. Un nodo 5/5 vale 1 bit; uno puro vale 0. El Gini es otra medida de impureza: $G = 1-\sum_k p_k^2$. La ganancia de un corte es
$$IG = H(\text{padre}) - \tfrac{N_\text{izq}}{N}H(\text{izq}) - \tfrac{N_\text{der}}{N}H(\text{der}),$$ donde $N$ es el número de ejemplos que llegan al nodo ($n$ sigue siendo el número de variables).

In [ ]:
animales = pd.DataFrame({
    "orejas":  ["puntiagudas", "puntiagudas", "puntiagudas", "caídas", "puntiagudas",
                "puntiagudas", "puntiagudas", "caídas", "caídas", "caídas"],
    "cara":    ["redonda", "redonda", "alargada", "redonda", "alargada",
                "redonda", "alargada", "redonda", "alargada", "alargada"],
    "bigotes": ["sí", "sí", "sí", "sí", "no", "no", "no", "no", "no", "no"],
    "peso_kg": [4.1, 3.6, 4.8, 3.2, 6.2, 7.5, 12.0, 6.0, 20.0, 9.0],
    "gato":    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
})
animales

**Antes de ejecutar.** Mira la tabla. ¿Con cuál de los tres rasgos (orejas, cara, bigotes) partirías primero? ¿Cuánto crees que vale la ganancia de ese corte: cerca de 0, cerca de 0.5 o cerca de 1 bit?

**Tu predicción:** _(escribe aquí)_

In [ ]:
def conteos(y):
    # conteos por clase en el orden [no gato, gato]
    y = np.asarray(y)
    return np.array([np.sum(y == 0), np.sum(y == 1)])


y_an = animales["gato"].to_numpy()
padre = conteos(y_an)
print(f"raíz: {padre}: H = {du.entropia(padre):.3f} bits · Gini = {du.gini(padre):.3f}")
filas = []
for rasgo, valor in [("orejas", "puntiagudas"), ("cara", "redonda"), ("bigotes", "sí")]:
    lado = (animales[rasgo] == valor).to_numpy()
    izq, der = conteos(y_an[lado]), conteos(y_an[~lado])
    filas.append({"rasgo": rasgo, "izq [no, sí]": f"{valor}: {izq}", "der [no, sí]": f"resto: {der}",
                  "H izq": du.entropia(izq) + 0.0, "H der": du.entropia(der) + 0.0,
                  "IG (bits)": du.ganancia_info(padre, izq, der),
                  "ganancia Gini": du.ganancia_info(padre, izq, der, impureza=du.gini)})
pd.DataFrame(filas).round(3)

**Bigotes** gana: $IG = 1 - \big[\tfrac{4}{10}\cdot 0 + \tfrac{6}{10}\cdot 0.650\big] = 0.61$ bits (es el mismo número que hicimos en la slide). La izquierda queda pura y la derecha casi pura. Orejas y cara apenas ordenan nada. Gini y entropía eligen el mismo rasgo; casi siempre coinciden, y Gini es un poco más barato de calcular porque no usa logaritmos.

Un árbol repite esto en cada nodo: prueba **todos** los cortes de **todas** las variables y se queda con el de mayor ganancia (búsqueda codiciosa). Con una variable numérica como `peso_kg`, los candidatos son los puntos medios entre valores consecutivos. Eso lo vas a programar en L3.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
p = np.linspace(0.001, 0.999, 300)
axs[0].plot(p, [du.entropia([1 - q, q]) for q in p], color=COLORES["w"], label="entropía (bits)")
axs[0].plot(p, [du.gini([1 - q, q]) for q in p], color=COLORES["lambda"], label="Gini")
axs[0].set(xlabel="proporción de gatos en el nodo", ylabel="impureza", title="Dos medidas de revoltura")
axs[0].legend()
for clase, marca, color in [(0, "o", COLORES["datos"]), (1, "X", COLORES["error"])]:
    sel = y_an == clase
    axs[1].scatter(animales["peso_kg"][sel], np.zeros(sel.sum()) + clase * 0.1, marker=marca, s=90,
                   color=color, edgecolors=COLORES["tinta"], label=["no gato", "gato"][clase])
axs[1].set(xlabel="peso (kg)", yticks=[], title="¿Dónde cortarías el peso?", ylim=(-0.3, 0.4))
axs[1].legend(fontsize=8)
plt.show()

> ### ✅ Reacción en Zoom · fin de L1
> Escribe en el chat la ganancia del corte por bigotes (dos decimales).
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L2 · Investiga: ¿qué tan profundo?
Volvemos al caso del Lab 2: ¿habrá un **flujo neto extremo** ($|$`future_net`$| \ge 6$, umbral congelado con train) en las próximas 3 h? Usamos las mismas variables (rezagos de 24 h o más, calendario y capacidad) más la hora y el día de la semana, que a un árbol le sirven tal cual.

La métrica es la **average precision** (AP) en validación. Un clasificador al azar tiene AP ≈ prevalencia (≈ 0.15).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import average_precision_score

df = pd.read_parquet(RUTA_DATOS)
tr = df[df["particion"] == "train"].copy()
va = df[df["particion"] == "val"].copy()
te = df[df["particion"] == "test"].copy()
UMBRAL = UMBRAL_EVENTO                            # celda L0: percentil 85 de |future_net| en train (Lab 2), congelado
assert UMBRAL == float(tr["future_net"].abs().quantile(0.85))
y_tr, y_va, y_te = ((d["future_net"].abs() >= UMBRAL).astype(int).to_numpy() for d in (tr, va, te))

HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]


def variables(d):
    # Lo que se sabe en la hora t sin mirar la hora t ni el futuro: rezagos ≥ 24 h, calendario y capacidad
    return pd.DataFrame({
        "net_lag_24h": d["net_lag_24h"], "net_lag_168h": d["net_lag_168h"], "net_mean_24h": d["net_mean_24h"],
        "abs_net_lag_24h": d["net_lag_24h"].abs(), "abs_net_lag_168h": d["net_lag_168h"].abs(),
        "abs_net_mean_24h": d["net_mean_24h"].abs(),
        "departures_mean_24h": d["departures_mean_24h"], "departures_mean_168h": d["departures_mean_168h"],
        "departures_lag_168h": d["departures_lag_168h"], "capacity": d["capacity"],
        "is_weekend": d["is_weekend"], "pico": d["hour_of_day"].isin(HORAS_PICO).astype(int),
        "hour_of_day": d["hour_of_day"], "day_of_week": d["day_of_week"],
    }, index=d.index)


X_tr, X_va, X_te = variables(tr), variables(va), variables(te)
print(f"X_tr: {X_tr.shape} · prevalencia val = {y_va.mean():.3f}")

# Precalculamos la curva una vez: 20 árboles sobre ~254 mil filas, de 40 s a 2.5 min en Colab (no la interrumpas)
PROFUNDIDADES = list(range(1, 21))
arboles, ap_tr, ap_va = {}, [], []
for d in PROFUNDIDADES:
    arboles[d] = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    ap_tr.append(average_precision_score(y_tr, arboles[d].predict_proba(X_tr)[:, 1]))
    ap_va.append(average_precision_score(y_va, arboles[d].predict_proba(X_va)[:, 1]))

**Antes de mover el slider.** Si la profundidad sube de 2 a 20, ¿qué le pasa a la AP de **train**? ¿Y a la de **validación**? Dibuja en tu cabeza las dos curvas.

**Tu predicción:** _(escribe aquí)_

In [ ]:
def explora_profundidad(max_depth=3):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
    axs[0].plot(PROFUNDIDADES, ap_tr, "o-", color=COLORES["w"], label="AP train")
    axs[0].plot(PROFUNDIDADES, ap_va, "s-", color=COLORES["error"], label="AP val")
    axs[0].axvline(max_depth, color=COLORES["amarillo"], lw=4, alpha=0.7)
    axs[0].axhline(y_va.mean(), color=COLORES["tinta"], ls=":", lw=1, label="azar ≈ prevalencia")
    i = PROFUNDIDADES.index(max_depth)
    axs[0].set(xlabel="max_depth", ylabel="AP", ylim=(0, 1.02),
               title=f"max_depth = {max_depth}: train {ap_tr[i]:.3f} · val {ap_va[i]:.3f}")
    axs[0].legend(fontsize=8)
    if max_depth <= 3:
        plot_tree(arboles[max_depth], feature_names=list(X_tr.columns), class_names=["no", "sí"],
                  filled=True, impurity=False, proportion=True, fontsize=7, ax=axs[1])
        axs[1].set_title("El árbol (proporciones por nodo)")
    else:
        imp = pd.Series(arboles[max_depth].feature_importances_, index=X_tr.columns).sort_values()
        axs[1].barh(imp.index, imp.values, color=COLORES["w"])
        axs[1].set(title="Importancia de cada variable (reducción de impureza)", xlabel="importancia")
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(explora_profundidad,
                     max_depth=widgets.IntSlider(value=3, min=1, max=20, step=1, description="max_depth"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Alternativa sin widget: cambia y re-ejecuta
explora_profundidad(max_depth=3)

La AP de train sube sin freno hasta 1.0 (un árbol sin límite memoriza cada estación-hora). La de validación sube, llega a su máximo alrededor de profundidad 10 (meseta entre 8 y 11) y luego **baja**. A la izquierda del máximo hay **sesgo alto** (el árbol es demasiado corto para el patrón); a la derecha, **varianza alta** (aprende ruido de junio y julio que agosto no repite). Es la misma curva de validación del Lab 2, ahora con `max_depth` en lugar del grado del polinomio.

> ### ✅ Reacción en Zoom · fin de L2
> Chat waterfall: escribe la profundidad que entregarías y su AP en val; envía con el "¡ya!".
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L3 · Completa las funciones (esto se califica)
Tres funciones que juntas son el corazón de un árbol: medir la impureza, medir cuánto la baja un corte y buscar el mejor corte de una variable numérica.

In [ ]:
import numpy as np


def entropia(conteos):
    """Entropía en BITS de un nodo a partir de sus conteos por clase.

    conteos : lista o arreglo de enteros ≥ 0, uno por clase (p. ej. [5, 5] o [3, 0, 1]).
    H = −Σ p_k log₂ p_k,  con p_k = conteo_k / total. Las clases con conteo 0 no aportan (0·log 0 = 0).
    Si el total es 0 (nodo vacío) regresa 0.0. Regresa un float.
    """
    c = np.asarray(conteos, dtype=float)

    # TODO: calcula las proporciones, descarta las que valen 0 y suma −p·log₂(p).
    raise NotImplementedError("Completa entropia (L3): busca el # TODO.")

In [ ]:
du.revisa_aislada(entropia, graded=['ganancia_info', 'mejor_corte'])
# Autocomprobación de entropia
assert np.isclose(entropia([5, 5]), 1.0), f"Un nodo 5/5 vale 1 bit; obtuve {entropia([5, 5])}"
assert entropia([7, 0]) == 0.0, "Un nodo puro vale 0 (¿calculaste log de 0?)"
assert entropia([0, 0]) == 0.0, "Un nodo vacío vale 0.0"
assert np.isclose(entropia([1, 5]), 0.6500224), f"H(1/6) ≈ 0.650 bits; obtuve {entropia([1, 5])}. ¿Usaste log₂?"
assert np.isclose(entropia([2, 2, 2, 2]), 2.0), "Con 4 clases iguales son 2 bits"
assert isinstance(entropia([3, 1]), float), "Regresa un float"
print("✅ entropia pasa la autocomprobación")

In [ ]:
def ganancia_info(padre, izq, der):
    """Ganancia de información (bits) de partir `padre` en `izq` y `der`.

    padre, izq, der : conteos por clase (mismo orden de clases en los tres).
    IG = H(padre) − (N_izq/N)·H(izq) − (N_der/N)·H(der),  con N = total de padre (ejemplos en el nodo).
    Usa tu función entropia. Si el padre está vacío regresa 0.0. Regresa un float.
    """
    padre = np.asarray(padre, dtype=float)
    izq = np.asarray(izq, dtype=float)
    der = np.asarray(der, dtype=float)

    # TODO: H(padre) menos el promedio de H(hijo) ponderado por el tamaño de cada hijo.
    raise NotImplementedError("Completa ganancia_info (L3): busca el # TODO.")

In [ ]:
du.revisa_aislada(ganancia_info, graded=['entropia', 'mejor_corte'])
# Autocomprobación de ganancia_info
assert np.isclose(ganancia_info([5, 5], [4, 0], [1, 5]), 0.6099865), "El corte por bigotes vale ≈ 0.61 bits"
assert np.isclose(ganancia_info([4, 4], [2, 2], [2, 2]), 0.0), "Si los hijos tienen las mismas proporciones que el padre, la ganancia es 0"
assert np.isclose(ganancia_info([5, 5], [5, 0], [0, 5]), 1.0), "Un corte perfecto de un nodo 5/5 gana 1 bit"
assert np.isclose(ganancia_info([5, 5], [1, 5], [4, 0]), ganancia_info([5, 5], [4, 0], [1, 5])), "Intercambiar izq y der no cambia la ganancia"
print("✅ ganancia_info pasa la autocomprobación")

In [ ]:
def mejor_corte(x, y):
    """Mejor umbral para UNA variable numérica, con ganancia de información (entropía en bits).

    x : (m,) valores numéricos (no necesariamente ordenados).   y : (m,) etiquetas de clase.
    Candidatos: los puntos medios entre valores DISTINTOS consecutivos de x ya ordenada.
    Cada candidato u manda a la izquierda los ejemplos con x <= u y a la derecha los demás.
    Regresa (umbral, ganancia) con el candidato de MAYOR ganancia (float, float).
    Regla de empate: si dos candidatos tienen la misma ganancia (diferencia < 1e-12), gana el umbral MÁS CHICO.
    Si no hay candidatos (todos los x son iguales) regresa (None, 0.0). Si hay candidatos pero todos dan
    ganancia 0 (p. ej. y es de una sola clase), regresa el más chico con ganancia 0.0: empieza con mejor ganancia = −∞.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y)

    # TODO: 1) conteos del padre por clase (np.unique(y)); 2) candidatos = puntos medios de np.unique(x);
    #       3) para cada candidato, de menor a mayor, cuenta la izquierda, la derecha es padre − izq,
    #          y calcula ganancia_info; 4) actualiza solo si la ganancia es mayor por más de 1e-12.
    raise NotImplementedError("Completa mejor_corte (L3): busca el # TODO.")

In [ ]:
du.revisa_aislada(mejor_corte, graded=['entropia', 'ganancia_info'])
# Autocomprobación de mejor_corte con el peso de los animales
u, g = mejor_corte(animales["peso_kg"], animales["gato"])
print(f"mejor corte del peso: peso <= {u} kg · ganancia = {g:.4f} bits")
assert np.isclose(u, 5.4) and np.isclose(g, 0.6099865), (
    "Hay dos cortes con 0.61 bits (5.4 y 6.85 kg); la regla de empate pide el más chico, 5.4. Obtuve " + str((u, g)))
assert mejor_corte([3.0, 3.0, 3.0], [0, 1, 0]) == (None, 0.0), "Sin candidatos regresa (None, 0.0)"
assert np.isclose(mejor_corte([4, 1, 3, 2], [1, 0, 1, 0])[0], 2.5), "x no viene ordenada: ordénala tú (np.unique ayuda)"

# Contra scikit-learn: un árbol de profundidad 1 con criterio de entropía elige el mismo umbral
rng = np.random.default_rng(3)
x_r = rng.normal(size=300)
y_r = (x_r + rng.normal(0, 0.8, 300) > 0.4).astype(int)
from sklearn.tree import DecisionTreeClassifier
tocon = DecisionTreeClassifier(max_depth=1, criterion="entropy").fit(x_r[:, None], y_r)
u_r, g_r = mejor_corte(x_r, y_r)
assert np.isclose(u_r, tocon.tree_.threshold[0]), f"sklearn corta en {tocon.tree_.threshold[0]:.4f} y tú en {u_r}"
print(f"✅ mejor_corte coincide con DecisionTreeClassifier(max_depth=1): umbral {u_r:.4f}, ganancia {g_r:.4f}")

**Fíjate en el empate.** Con el peso hay dos cortes de 0.61 bits: $\le 5.4$ kg deja (4 gatos, 0) a la izquierda y $\le 6.85$ kg deja (5 gatos, 1 perro). Las dos particiones son distintas y valen lo mismo; una regla de empate fija es lo que hace que tu función sea reproducible. scikit-learn, por cierto, desempata al azar entre variables (por eso tiene `random_state`).

> ### ✅ Reacción en Zoom · fin de L3
> Reacciona con ✅ cuando las tres autocomprobaciones digan ✅.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L4 · Caza la fuga: una variable que ve el futuro
Una compañera dice que el flujo neto por hora es muy ruidoso y lo **suavizó** con una media móvil de 7 horas por estación. Agregó esa variable al árbol de profundidad 8 y presume una AP en validación muy por encima de la tuya. Ejecuta su código.

In [ ]:
# ---- pipeline de la compañera (no lo copies) ----
todo = df.sort_values(["station_id", "hour"]).copy()
todo["net_suave_7h"] = (todo.groupby("station_id")["net_flow"]
                        .transform(lambda s: s.rolling(7, center=True, min_periods=1).mean()))
todo = todo.sort_index()
X_tr_bug = variables(tr).assign(abs_net_suave_7h=todo.loc[tr.index, "net_suave_7h"].abs())
X_va_bug = variables(va).assign(abs_net_suave_7h=todo.loc[va.index, "net_suave_7h"].abs())

arbol_bug = DecisionTreeClassifier(max_depth=8, random_state=0).fit(X_tr_bug, y_tr)
ap_bug = average_precision_score(y_va, arbol_bug.predict_proba(X_va_bug)[:, 1])
print(f"AP val con la variable suavizada: {ap_bug:.3f}   ·   tu árbol de profundidad 8: {ap_va[PROFUNDIDADES.index(8)]:.3f}")

**La prueba del sospechoso.** Una forma rápida de auditar variables: usa cada una **sola** como puntaje y mide su AP en validación. Si una variable sola le gana a todo tu modelo, desconfía antes de celebrar.

In [ ]:
sospechosos = pd.Series({col: max(average_precision_score(y_va, X_va_bug[col]), average_precision_score(y_va, -X_va_bug[col]))
                         for col in X_va_bug.columns}).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
colores = [COLORES["error"] if col == "abs_net_suave_7h" else COLORES["w"] for col in sospechosos.index]
ax.barh(sospechosos.index, sospechosos.values, color=colores)
ax.axvline(y_va.mean(), color=COLORES["tinta"], ls=":", lw=1)
ax.set(xlabel="AP en val usando solo esa variable", title="Una variable sola contra la etiqueta")
plt.show()

**¿Dónde está la fuga y cómo la corriges?** Pista: ¿qué horas promedia `rolling(7, center=True)`? Compáralas con la ventana de `future_net`.

**Tu respuesta:** _(escribe aquí)_

> ### ✅ Reacción en Zoom · fin de L4
> Escribe en el chat qué horas promedia la variable suavizada.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L5 · Modifica y justifica: árbol vs bosque vs boosting
Tres apuestas sobre los mismos datos, sin fugas:
- **Árbol** con la profundidad que gana en validación (L2).
- **Bosque aleatorio:** muchos árboles (aquí con hojas de al menos 20 ejemplos), cada uno con una muestra *bootstrap* y un subconjunto de variables en cada corte; se **promedian**. Ataca la **varianza**.
- **Boosting** (`HistGradientBoostingClassifier`): árboles chicos en secuencia, cada uno corrige lo que los anteriores fallaron; se **suman**. Ataca el **sesgo**. Es la misma idea que XGBoost (Ng, C2 S4), ya incluida en scikit-learn.

La profundidad del árbol sale de validación (L2); el bosque y el boosting van con valores razonables fijos para que la celda quepa en el tiempo del lab (en el proyecto sí les harás rejilla). Test se toca **una vez**, al final. La celda tarda de 1 a 3 min en Colab gratuito.

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

prof_mejor = PROFUNDIDADES[int(np.argmax(ap_va))]
modelos = {
    f"árbol (max_depth={prof_mejor})": DecisionTreeClassifier(max_depth=prof_mejor, random_state=0),
    "bosque aleatorio (200 árboles)": RandomForestClassifier(n_estimators=200, min_samples_leaf=20, max_features="sqrt",
                                                             n_jobs=2, random_state=0),
    "boosting (HistGradientBoosting)": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                                                      early_stopping=False, random_state=0),
}
p_va, filas = {}, []
for nombre, modelo in modelos.items():
    t0 = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    segundos = time.perf_counter() - t0
    p_va[nombre] = modelo.predict_proba(X_va)[:, 1]
    filas.append({"modelo": nombre, "AP train": average_precision_score(y_tr, modelo.predict_proba(X_tr)[:, 1]),
                  "AP val": average_precision_score(y_va, p_va[nombre]), "segundos": segundos})
tabla = pd.DataFrame(filas).set_index("modelo")
tabla.round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for (nombre, p), color in zip(p_va.items(), [COLORES["lambda"], COLORES["yhat"], COLORES["w"]]):
    prec, rec, _ = precision_recall_curve(y_va, p)
    ax.plot(rec, prec, color=color, label=f"{nombre} · AP {tabla.loc[nombre, 'AP val']:.3f}")
ax.axhline(y_va.mean(), color=COLORES["tinta"], ls=":", lw=1, label="azar (prevalencia)")
ax.set(xlabel="recall", ylabel="precision", title="Curva PR en validación", xlim=(0, 1), ylim=(0, 1))
ax.legend(fontsize=8)
plt.show()

**Umbral con presupuesto.** Igual que en el Lab 2: la cuadrilla atiende como máximo el **25 %** de las estaciones-hora. Elegimos en validación el umbral más bajo que respeta ese presupuesto, para el modelo con mayor AP en validación.

In [ ]:
from sklearn.metrics import precision_score, recall_score


def umbral_presupuesto(p, presupuesto):
    # El umbral más bajo t (entre los valores de p) con mean(p >= t) <= presupuesto
    candidatos = np.unique(p)                                             # ordenados de menor a mayor
    tasa = 1 - np.searchsorted(np.sort(p), candidatos, side="left") / len(p)   # mean(p >= t) de cada candidato
    return float(candidatos[np.argmax(tasa <= presupuesto)])


PRESUPUESTO = 0.25
ganador = tabla["AP val"].idxmax()
umbral_clf = umbral_presupuesto(p_va[ganador], PRESUPUESTO)
alerta_va = p_va[ganador] >= umbral_clf
print(f"ganador en val: {ganador}")
print(f"umbral = {umbral_clf:.4f} · val: alertas = {alerta_va.mean():.1%} · precision = {precision_score(y_va, alerta_va):.3f} · recall = {recall_score(y_va, alerta_va):.3f}")

# Una sola vez, con el modelo y el umbral congelados:
p_te = modelos[ganador].predict_proba(X_te)[:, 1]
alerta_te = p_te >= umbral_clf
print(f"test: AP = {average_precision_score(y_te, p_te):.3f} · alertas = {alerta_te.mean():.1%} · "
      f"precision = {precision_score(y_te, alerta_te):.3f} · recall = {recall_score(y_te, alerta_te):.3f}")

**Justifica (O4.2).** Con la tabla: ¿qué modelo tiene la brecha train–val más grande? ¿El bosque le gana al árbol por bajar sesgo o por bajar varianza? ¿Y el boosting? Cita los números.

**Tu respuesta:** _(escribe aquí)_

> ### ✅ Reacción en Zoom · fin de L5
> Chat waterfall: escribe el modelo ganador y su AP en val.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L6 · Mini-reto (abierto): ¿qué despliegas y con qué umbral?
Imagina que eres quien decide. Propón un modelo y un umbral para la operación de rebalanceo y justifícalo en 3 líneas. Ideas para explorar: `min_samples_leaf` o `max_features` en el bosque, `learning_rate` y `max_iter` en el boosting, o la importancia de las variables (¿de qué depende el modelo?). Recuerda: todo se decide en validación.

No se califica: comparte tu hallazgo en el chat.

In [ ]:
# Tu exploración aquí

---
### Antes de entregar
1. *Reiniciar y ejecutar todo*. Las autocomprobaciones de L3 deben decir ✅.
2. Descarga el notebook (*Archivo*, *Descargar*, *Descargar .ipynb*) y súbelo a Moodle. No cambies el nombre de las funciones `graded`.